# 014 — ReAct Agent with Tool Calling (LangGraph)
**Agentic Orchestration** | Reason + Act loop

**ReAct** (Reason + Act) interleaves:
- **Thought**: what do I need to do next?
- **Action**: call a tool
- **Observation**: tool result
- **Repeat** until the answer is ready

Tools in this notebook: calculator · Wikipedia lookup · RAG search · text stats


In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── Define tools ─────────────────────────────────────────────────────────
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
import math, re

# ── RAG knowledge base ────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
_docs = [Document(page_content=c)
         for text in [ai_text, ml_text, dl_text, nlp_text, rag_text]
         for c in splitter.split_text(text[:15000])]
_store = FAISS.from_documents(_docs, embeddings)

@tool
def rag_search(query: str) -> str:
    '''Search the local RAG knowledge base about AI, ML, NLP, deep learning.'''
    docs = _store.similarity_search(query, k=3)
    return "\n\n".join(f"[{i+1}] {d.page_content[:300]}" for i, d in enumerate(docs))

@tool
def calculator(expression: str) -> str:
    '''Evaluate a simple mathematical expression. E.g., "2 ** 10", "100 / 4 + 3".'''
    try:
        # safe eval: only allow numbers and math operators
        safe_expr = re.sub(r"[^0-9+\-*/().,\s]", "", expression)
        result = eval(safe_expr, {"__builtins__": {}, "math": math})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def text_stats(text: str) -> str:
    '''Return word count, sentence count, and char count of a given text.'''
    words = len(text.split())
    sents = len([s for s in text.split(".") if s.strip()])
    chars = len(text)
    return f"words={words}, sentences={sents}, chars={chars}"

tools = [rag_search, calculator, text_stats]
print("✓ Tools:", [t.name for t in tools])


In [ ]:
# ── LangGraph ReAct agent ────────────────────────────────────────────────
from langgraph.prebuilt import create_react_agent

# Bind tools to the fast LLM
llm_with_tools = llm.bind_tools(tools)

# create_react_agent handles the ReAct loop automatically
react_app = create_react_agent(llm, tools)
print("✓ ReAct agent ready")


In [ ]:
# ── Test the agent ────────────────────────────────────────────────────────
from langchain_core.messages import HumanMessage

def run_agent(question: str):
    result = react_app.invoke({"messages": [HumanMessage(content=question)]})
    messages = result["messages"]

    print(f"Q: {question}")
    for msg in messages:
        role = type(msg).__name__.replace("Message", "")
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [Tool call] {tc['name']}({tc['args']})")
        elif role == "Tool":
            content_preview = str(msg.content)[:200]
            print(f"  [Tool result] {content_preview}...")
        elif role == "AI" and msg.content:
            print(f"  [Answer] {msg.content[:400]}")
    print()

run_agent("What is transformer architecture in deep learning? Give a brief explanation.")
run_agent("How many words are in this sentence: The quick brown fox jumps over the lazy dog")
run_agent("What is 2 to the power of 16?")


In [ ]:
# ── Multi-step reasoning demo ─────────────────────────────────────────────
run_agent(
    "Search for information about reinforcement learning, then tell me "
    "how many words are in the first retrieved passage."
)


## Key Takeaways
- `create_react_agent` from `langgraph.prebuilt` is the easiest way to build a ReAct loop
- The loop runs until the LLM produces a final answer (no tool calls)
- Tool definitions with `@tool` decorator auto-generate the JSON schema the LLM needs
- For production: add memory, retry logic, and tool timeouts
